# Results

In [1]:
import torch
import timm
import tome
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import eval_cpu
from eval_gpu import evaluate
from SReT import SReT_T_distill
import PiT_ToMe
import SReT_ToMe

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Current GPU Device: {torch.cuda.current_device()}")

batch_size = 128 # batch size doesn't matter since it's only used for acc evaluations
dataset_dir = '/media/datasets/imagenet/val' # path to imagenet val dataset
dataset_transform = transforms.Compose([
    transforms.Resize(256), 
    transforms.CenterCrop(224), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
dataset = datasets.ImageFolder(dataset_dir, transform=dataset_transform)
dataset_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True) 
print(f'Loaded {len(dataset)} ImageNet-1k Validation images')

CUDA available: True
GPU Device Name: NVIDIA GeForce RTX 4060 Ti
Current GPU Device: 0
Loaded 50000 ImageNet-1k Validation images


#### DeiT-Tiny-Distill 

In [2]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.40 %
Total Parameters:                           5.91 M
Theoretical FLOPs:                          2.17 G
Throughput (BS=128):            1825.88 images/sec
Throughput (BS=64):             1961.84 images/sec
Throughput (BS=32):             2046.70 images/sec
Throughput (BS=16):             2381.86 images/sec
Throughput (BS=1):               730.20 images/sec
Peak Activation Memory (BS=128):         227.20 MB
Peak Activation Memory (BS=64):          113.03 MB
Peak Activation Memory (BS=32):           56.44 MB
Peak Activation Memory (BS=16):           28.59 MB
Peak Activation Memory (BS=1):             1.76 MB



In [3]:
_ = eval_cpu.evaluate("deit")

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.05 ms
Throughput:                         141.79 img/sec



#### DeiT-Tiny-Distill + ToMe | Constant Reduction

In [4]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
tome.patch.timm(model, prop_attn=True)
model = model.cuda().eval()
model.r = 10
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            73.47 %
Total Parameters:                           5.91 M
Theoretical FLOPs:                          1.51 G
Throughput (BS=128):            2416.56 images/sec
Throughput (BS=64):             2626.23 images/sec
Throughput (BS=32):             2675.22 images/sec
Throughput (BS=16):             2632.75 images/sec
Throughput (BS=1):               250.33 images/sec
Peak Activation Memory (BS=128):         232.62 MB
Peak Activation Memory (BS=64):          117.20 MB
Peak Activation Memory (BS=32):           58.93 MB
Peak Activation Memory (BS=16):           29.64 MB
Peak Activation Memory (BS=1):             1.82 MB



In [5]:
_ = eval_cpu.evaluate("deit+tome+c", constant_r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.36 ms
Throughput:                         135.82 img/sec



In [6]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
tome.patch.timm(model, prop_attn=True)
model = model.cuda().eval()
model.r = 20
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            61.49 %
Total Parameters:                           5.91 M
Theoretical FLOPs:                          0.92 G
Throughput (BS=128):            4011.18 images/sec
Throughput (BS=64):             4227.83 images/sec
Throughput (BS=32):             4183.96 images/sec
Throughput (BS=16):             3692.11 images/sec
Throughput (BS=1):               249.82 images/sec
Peak Activation Memory (BS=128):         216.16 MB
Peak Activation Memory (BS=64):          110.27 MB
Peak Activation Memory (BS=32):           55.12 MB
Peak Activation Memory (BS=16):           27.62 MB
Peak Activation Memory (BS=1):             1.69 MB



In [7]:
_ = eval_cpu.evaluate("deit+tome+c", constant_r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   5.37 ms
Throughput:                         186.05 img/sec



#### PiT-Tiny-Distill

In [11]:
model = timm.create_model("pit_ti_distilled_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.42 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          1.01 G
Throughput (BS=128):            1799.83 images/sec
Throughput (BS=64):             1866.28 images/sec
Throughput (BS=32):             1951.04 images/sec
Throughput (BS=16):             2096.04 images/sec
Throughput (BS=1):               684.83 images/sec
Peak Activation Memory (BS=128):        1203.36 MB
Peak Activation Memory (BS=64):          601.51 MB
Peak Activation Memory (BS=32):          301.17 MB
Peak Activation Memory (BS=16):          150.38 MB
Peak Activation Memory (BS=1):             9.40 MB



In [12]:
_ = eval_cpu.evaluate("pit")

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   6.18 ms
Throughput:                         161.93 img/sec



#### PiT-Tiny-Distill + ToMe | Constant Reduction

In [7]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="constant", constant_r=10)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            73.77 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.80 G
Throughput (BS=128):            1569.31 images/sec
Throughput (BS=64):             1618.44 images/sec
Throughput (BS=32):             1658.93 images/sec
Throughput (BS=16):             1658.14 images/sec
Throughput (BS=1):               220.37 images/sec
Peak Activation Memory (BS=128):        1192.65 MB
Peak Activation Memory (BS=64):          596.16 MB
Peak Activation Memory (BS=32):          298.50 MB
Peak Activation Memory (BS=16):          149.04 MB
Peak Activation Memory (BS=1):             9.32 MB



In [8]:
_ = eval_cpu.evaluate("pit+tome+c", constant_r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   8.37 ms
Throughput:                         119.51 img/sec



In [9]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="constant", constant_r=20)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            71.09 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.66 G
Throughput (BS=128):            1724.93 images/sec
Throughput (BS=64):             1789.32 images/sec
Throughput (BS=32):             1813.64 images/sec
Throughput (BS=16):             1759.49 images/sec
Throughput (BS=1):               222.73 images/sec
Peak Activation Memory (BS=128):        1192.65 MB
Peak Activation Memory (BS=64):          596.16 MB
Peak Activation Memory (BS=32):          298.50 MB
Peak Activation Memory (BS=16):          149.04 MB
Peak Activation Memory (BS=1):             9.32 MB



In [10]:
_ = eval_cpu.evaluate("pit+tome+c", constant_r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.84 ms
Throughput:                         127.55 img/sec



#### PiT-Tiny-Distill + ToMe | Linear Reduction

In [3]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="linear", linear_r=10)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.06 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.86 G
Throughput (BS=128):            1578.59 images/sec
Throughput (BS=64):             1635.06 images/sec
Throughput (BS=32):             1678.89 images/sec
Throughput (BS=16):             1664.36 images/sec
Throughput (BS=1):               235.83 images/sec
Peak Activation Memory (BS=128):        1192.65 MB
Peak Activation Memory (BS=64):          596.16 MB
Peak Activation Memory (BS=32):          298.50 MB
Peak Activation Memory (BS=16):          149.04 MB
Peak Activation Memory (BS=1):             9.32 MB



In [4]:
_ = eval_cpu.evaluate("pit+tome+l", linear_r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   8.27 ms
Throughput:                         120.95 img/sec



In [5]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="linear", linear_r=20)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            71.13 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.71 G
Throughput (BS=128):            1792.73 images/sec
Throughput (BS=64):             1857.88 images/sec
Throughput (BS=32):             1870.69 images/sec
Throughput (BS=16):             1797.43 images/sec
Throughput (BS=1):               230.10 images/sec
Peak Activation Memory (BS=128):        1192.65 MB
Peak Activation Memory (BS=64):          596.16 MB
Peak Activation Memory (BS=32):          298.50 MB
Peak Activation Memory (BS=16):          149.04 MB
Peak Activation Memory (BS=1):             9.32 MB



In [6]:
_ = eval_cpu.evaluate("pit+tome+l", linear_r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   7.63 ms
Throughput:                         131.15 img/sec



#### PiT-Tiny-Distill + ToMe | Exponential Reduction

In [13]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="exponential", initial_r=0.25, alpha=0.0)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            72.03 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.80 G
Throughput (BS=128):            1914.08 images/sec
Throughput (BS=64):             2006.74 images/sec
Throughput (BS=32):             2096.31 images/sec
Throughput (BS=16):             2081.23 images/sec
Throughput (BS=1):               369.41 images/sec
Peak Activation Memory (BS=128):        1192.65 MB
Peak Activation Memory (BS=64):          596.16 MB
Peak Activation Memory (BS=32):          298.50 MB
Peak Activation Memory (BS=16):          149.04 MB
Peak Activation Memory (BS=1):             9.32 MB



In [2]:
_ = eval_cpu.evaluate("pit+tome+e", initial_r=0.25, alpha=0)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   6.56 ms
Throughput:                         152.35 img/sec



In [3]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="exponential", initial_r=0.40, alpha=0.2)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            62.63 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.63 G
Throughput (BS=128):            2351.17 images/sec
Throughput (BS=64):             2453.14 images/sec
Throughput (BS=32):             2458.11 images/sec
Throughput (BS=16):             2365.11 images/sec
Throughput (BS=1):               285.97 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [4]:
_ = eval_cpu.evaluate("pit+tome+e", initial_r=0.40, alpha=0.2)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   6.34 ms
Throughput:                         157.79 img/sec



In [5]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, schedule_type="exponential", initial_r=0.10, alpha=0.6)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            73.90 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.88 G
Throughput (BS=128):            1620.17 images/sec
Throughput (BS=64):             1660.17 images/sec
Throughput (BS=32):             1717.03 images/sec
Throughput (BS=16):             1717.70 images/sec
Throughput (BS=1):               236.41 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



In [6]:
_ = eval_cpu.evaluate("pit+tome+e", initial_r=0.1, alpha=0.6)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                   8.18 ms
Throughput:                         122.31 img/sec



#### SReT-Tiny-Distill

In [8]:
model = SReT_T_distill(pretrained=False)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            77.42 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.91 G
Throughput (BS=128):            1072.86 images/sec
Throughput (BS=64):             1169.82 images/sec
Throughput (BS=32):             1248.20 images/sec
Throughput (BS=16):             1298.30 images/sec
Throughput (BS=1):               223.58 images/sec
Peak Activation Memory (BS=128):         795.76 MB
Peak Activation Memory (BS=64):          397.88 MB
Peak Activation Memory (BS=32):          200.88 MB
Peak Activation Memory (BS=16):          100.44 MB
Peak Activation Memory (BS=1):             6.22 MB



In [9]:
_ = eval_cpu.evaluate("sret")

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  12.00 ms
Throughput:                          83.32 img/sec



#### SReT-Tiny-Distill + ToMe | Constant Reduction

In [2]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="constant", constant_r=10)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            71.01 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.32 G
Throughput (BS=128):            1176.27 images/sec
Throughput (BS=64):             1277.75 images/sec
Throughput (BS=32):             1326.16 images/sec
Throughput (BS=16):             1168.58 images/sec
Throughput (BS=1):               110.24 images/sec
Peak Activation Memory (BS=128):         769.79 MB
Peak Activation Memory (BS=64):          385.02 MB
Peak Activation Memory (BS=32):          192.51 MB
Peak Activation Memory (BS=16):           96.50 MB
Peak Activation Memory (BS=1):             6.01 MB



In [3]:
_ = eval_cpu.evaluate("sret+tome+c", constant_r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  13.16 ms
Throughput:                          75.97 img/sec



In [4]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="constant", constant_r=20)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            41.06 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.06 G
Throughput (BS=128):            1331.23 images/sec
Throughput (BS=64):             1439.60 images/sec
Throughput (BS=32):             1476.42 images/sec
Throughput (BS=16):             1181.77 images/sec
Throughput (BS=1):               112.56 images/sec
Peak Activation Memory (BS=128):         756.22 MB
Peak Activation Memory (BS=64):          380.27 MB
Peak Activation Memory (BS=32):          189.15 MB
Peak Activation Memory (BS=16):           94.95 MB
Peak Activation Memory (BS=1):             5.91 MB



In [5]:
_ = eval_cpu.evaluate("sret+tome+c", constant_r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  12.03 ms
Throughput:                          83.16 img/sec



#### SReT-Tiny-Distill + ToMe | Linear Reduction

In [6]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="linear", linear_r=10)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.64 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.46 G
Throughput (BS=128):            1177.32 images/sec
Throughput (BS=64):             1279.34 images/sec
Throughput (BS=32):             1330.01 images/sec
Throughput (BS=16):             1207.77 images/sec
Throughput (BS=1):               116.12 images/sec
Peak Activation Memory (BS=128):         756.22 MB
Peak Activation Memory (BS=64):          380.27 MB
Peak Activation Memory (BS=32):          189.15 MB
Peak Activation Memory (BS=16):           94.95 MB
Peak Activation Memory (BS=1):             5.91 MB



In [7]:
_ = eval_cpu.evaluate("sret+tome+l", linear_r=10)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  13.69 ms
Throughput:                          73.06 img/sec



In [8]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="linear", linear_r=20)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            14.87 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.07 G
Throughput (BS=128):            1427.14 images/sec
Throughput (BS=64):             1538.98 images/sec
Throughput (BS=32):             1553.07 images/sec
Throughput (BS=16):             1235.32 images/sec
Throughput (BS=1):               117.21 images/sec
Peak Activation Memory (BS=128):         729.46 MB
Peak Activation Memory (BS=64):          366.76 MB
Peak Activation Memory (BS=32):          183.15 MB
Peak Activation Memory (BS=16):           91.89 MB
Peak Activation Memory (BS=1):             5.70 MB



In [9]:
_ = eval_cpu.evaluate("sret+tome+l", linear_r=20)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  11.72 ms
Throughput:                          85.30 img/sec



#### SReT-Tiny-Distill + ToMe | Exponential Reduction

In [10]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="exponential", initial_r=0.25, alpha=0.0)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            75.95 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.49 G
Throughput (BS=128):            1368.67 images/sec
Throughput (BS=64):             1511.36 images/sec
Throughput (BS=32):             1591.05 images/sec
Throughput (BS=16):             1599.63 images/sec
Throughput (BS=1):               174.37 images/sec
Peak Activation Memory (BS=128):         489.38 MB
Peak Activation Memory (BS=64):          245.43 MB
Peak Activation Memory (BS=32):          123.13 MB
Peak Activation Memory (BS=16):           62.34 MB
Peak Activation Memory (BS=1):             3.90 MB



In [2]:
_ = eval_cpu.evaluate("sret+tome+e", initial_r=0.25, alpha=0)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  11.88 ms
Throughput:                          84.15 img/sec



In [12]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="exponential", initial_r=0.40, alpha=0.2)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            69.71 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.13 G
Throughput (BS=128):            1886.90 images/sec
Throughput (BS=64):             2050.53 images/sec
Throughput (BS=32):             2077.24 images/sec
Throughput (BS=16):             1860.68 images/sec
Throughput (BS=1):               147.91 images/sec
Peak Activation Memory (BS=128):         391.51 MB
Peak Activation Memory (BS=64):          195.76 MB
Peak Activation Memory (BS=32):           97.88 MB
Peak Activation Memory (BS=16):           48.94 MB
Peak Activation Memory (BS=1):             3.90 MB



In [2]:
_ = eval_cpu.evaluate("sret+tome+e", initial_r=0.40, alpha=0.2)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  11.35 ms
Throughput:                          88.12 img/sec



In [4]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, schedule_type="exponential", initial_r=0.1, alpha=0.6)
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            76.35 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.61 G
Throughput (BS=128):            1186.62 images/sec
Throughput (BS=64):             1290.96 images/sec
Throughput (BS=32):             1354.81 images/sec
Throughput (BS=16):             1334.67 images/sec
Throughput (BS=1):               128.30 images/sec
Peak Activation Memory (BS=128):         664.75 MB
Peak Activation Memory (BS=64):          334.00 MB
Peak Activation Memory (BS=32):          166.50 MB
Peak Activation Memory (BS=16):           84.50 MB
Peak Activation Memory (BS=1):             5.19 MB



In [5]:
_ = eval_cpu.evaluate("sret+tome+e", initial_r=0.10, alpha=0.6)

Target Batch Size:                               1
--------------------------------------------------
Latency:                                  13.08 ms
Throughput:                          76.44 img/sec

